read data 

In [0]:
crm = spark.read.format("delta").load("/Workspace/Cross-System Data Monitoring Platform/silver/crm")

billing = spark.read.format("delta").load("/Workspace/Cross-System Data Monitoring Platform/silver/billing")

analytics = spark.read.format("delta").load("/Workspace/Cross-System Data Monitoring Platform/silver/analytics")

trust = spark.read.format("delta").load("/Workspace/Cross-System Data Monitoring Platform/gold/trust_score")

missing = spark.read.format("delta").load("/Workspace/Cross-System Data Monitoring Platform/gold/missing_records")

duplicates = spark.read.format("delta").load("/Workspace/Cross-System Data Monitoring Platform/gold/duplicates")

drift = spark.read.format("delta").load("/Workspace/Cross-System Data Monitoring Platform/gold/drift_report")

printing data from delta table

In [0]:
crm.show(5)

billing.show(5)

analytics.show(5)

trust.show()

missing.show()

duplicates.show()

drift.show()

+-----------+------------+--------------------+-----------+----------+
|customer_id|        name|               email|signup_date|      city|
+-----------+------------+--------------------+-----------+----------+
|  CRM009746|    Sai Shah|sai.shah35@gmail.com| 2023-07-14|   XX_CITY|
|  CRM007145| Rohan Mehta|rohan.mehta565@ya...| 2023-11-21|    Nagpur|
|  CRM002099|  Aadhya Rao|aadhya.rao716@hot...| 2024-02-09|Coimbatore|
|  CRM008456|  Sneha Bose|sneha.bose955@red...| 2022-11-21|   Chennai|
|  CRM007689|Reyan Tiwari|                NULL| 2022-01-28| Bangalore|
+-----------+------------+--------------------+-----------+----------+
only showing top 5 rows
+--------------+-----------+-------+----------------+---------+
|transaction_id|customer_id| amount|transaction_date|   status|
+--------------+-----------+-------+----------------+---------+
|    TXN0005938|  CRM000702|1933.03|      2024-02-18|completed|
|    TXN0001759|  CRM004718| 345.62|      2023-12-26|  pending|
|    TXN0004538| 

summary of delta table

In [0]:
crm_count = crm.count()

billing_count = billing.count()

analytics_count = analytics.count()

missing_count = missing.count()

duplicate_count = duplicates.count()

drift_count = drift.count()

trust table

In [0]:
trust.show()

trust_score = trust.collect()[0]["trust_score"]

+-----------+
|trust_score|
+-----------+
|       36.6|
+-----------+



creating dashboard

In [0]:
dashboard = [
    ("CRM Records", float(crm_count)),
    ("Billing Records", float(billing_count)),
    ("Analytics Records", float(analytics_count)),
    ("Missing Records", float(missing_count)),
    ("Duplicate Records", float(duplicate_count)),
    ("Drift Records", float(drift_count)),
    ("Trust Score", float(trust_score))   ]

dashboard_df = spark.createDataFrame( dashboard, ["Metric", "Value"])

dashboard_df.show(truncate=False)


+-----------------+-------+
|Metric           |Value  |
+-----------------+-------+
|CRM Records      |10500.0|
|Billing Records  |11680.0|
|Analytics Records|912.0  |
|Missing Records  |3959.0 |
|Duplicate Records|0.0    |
|Drift Records    |912.0  |
|Trust Score      |36.6   |
+-----------------+-------+



change dashboard type int to float

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("Metric", StringType(), True),
    StructField("Value", DoubleType(), True)
])

dashboard_df = spark.createDataFrame(dashboard, schema)

dashboard_df.show(truncate=False)

+-----------------+-------+
|Metric           |Value  |
+-----------------+-------+
|CRM Records      |10500.0|
|Billing Records  |11680.0|
|Analytics Records|912.0  |
|Missing Records  |3959.0 |
|Duplicate Records|0.0    |
|Drift Records    |912.0  |
|Trust Score      |36.6   |
+-----------------+-------+



saving dashboard data into delta table

In [0]:
dashboard_df.write \
.format("delta") \
.mode("overwrite") \
.save("/Workspace/Cross-System Data Monitoring Platform/gold/dashboard_data")

In [0]:
dashboard = spark.read.format("delta").load("/Workspace/Cross-System Data Monitoring Platform/gold/dashboard_data")

dashboard.show(truncate=False)

+-----------------+-------+
|Metric           |Value  |
+-----------------+-------+
|CRM Records      |10500.0|
|Billing Records  |11680.0|
|Analytics Records|912.0  |
|Missing Records  |3959.0 |
|Duplicate Records|0.0    |
|Drift Records    |912.0  |
|Trust Score      |36.6   |
+-----------------+-------+



creating dashboard into csv file

In [0]:
dashboard_df.toPandas().to_csv( "/Workspace/Cross-System Data Monitoring Platform/dashboard/dashboard.csv", index=False)

In [0]:
print("Dashboard Created Successfully")

print("Total Metrics :", dashboard.count())

Dashboard Created Successfully
Total Metrics : 7
